# w10-2. 학습한 모델 저장하고 다시 사용하기

**오늘 할 일**
1. 왜 모델을 저장해야 하는지 이해
2. **Logistic Regression**을 예시로 저장 → 불러오기 → 새 분자 예측 (worked example)
3. 같은 방식으로 **ANN, SVM, Decision Tree, Random Forest** 저장하기 (✏️ Grid search를 통해 가장 좋은 하이퍼파라미터를 확인한 후, 모델을 저장)

**데이터**: `skin_irritation_2Ddesc.csv` (지난 시간과 동일)

## 🚨 핵심 포인트

| 모델 | 저장해야 할 것 |
|---|---|
| Logistic Regression / ANN / SVM | feature 이름 + **scaler** + 모델 |
| Decision Tree / Random Forest | feature 이름 + 모델 (scaler 불필요) |

> ⚠️ normalization을 쓴 모델은 **학습 때 fit한 scaler**를 그대로 다시 써야 해.
> 새 분자에 `fit_transform`을 새로 부르면 **다른 평균/표준편차**로 변환되어 모델이 망가져.

---
# Part A. 이제까지 여러가지 실험을 진행한 이유?

- **새로운 분자 구조**를 예측할 때 사용할, 모델을 확정짓기 위함.
- 여러가지 실험을 통해 모델 성능을 확인하고, 가장 성능이 높은 모델을 최종 모델로 확정 및 배포. (다른 사람들도 사용할 수 있게 공유)

파이썬에서 모델을 저장하는 표준 도구는 **`joblib`**.

```python
import joblib
joblib.dump(객체, '파일이름.joblib')   # 저장
객체 = joblib.load('파일이름.joblib')   # 불러오기
```

---
# Part B. Logistic Regression — worked example

## B-1. 데이터 준비 + 모델 학습

지난 시간(w9-1) 코드 그대로.

## 🚨 가장 중요한 약속 — 학습과 예측은 같은 전처리

Logistic Regression은 **normalization 된 feature**로 학습한다.
→ 그래서 나중에 **새 분자에 예측할 때도 반드시 normalization 된 feature**를 넣어야 한다.
→ normalization을 다시 fit하면 평균/표준편차가 달라져서 망가지므로, **학습 때 fit한 scaler를 그대로 재사용**해야 한다.

```
[학습] X_sel  --(scaler.fit_transform)-->  X_scaled  --(clf.fit)-->  모델
[예측] new_X  --(scaler.transform)------>  X_scaled  --(clf.predict)-->  결과
                  ⬆ 같은 scaler! fit 다시 하면 안 됨
```

그래서 학습이 끝나면 `cols`, **`scaler`**, `clf` 세 개를 모두 손에 들고 있어야 함. 하나라도 빠지면 새 분자 예측 불가능.

In [1]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 데이터 불러오기
df = pd.read_csv('skin_irritation_2Ddesc.csv')
y = df['label']
X = df.drop(columns=['Chemical_Name', 'standardized_smi', 'label'])

# NaN 열 제거 + 저분산 열 제거
X = X.dropna(axis=1)
X = X.loc[:, X.std() >= 0.01]

# 상위 10개 descriptor 선택
selector = SelectKBest(f_classif, k=10)
selector.fit(X, y)
cols = X.columns[selector.get_support()]   # 골라진 feature 이름들
X_sel = X[cols]
print('선택된 descriptor:', list(cols))

# ⭐ Normalization — Logistic Regression은 scale에 민감하므로 필수
# fit_transform: 학습 데이터의 평균/표준편차를 '계산(fit)'하고 동시에 '변환(transform)'
# scaler 객체에는 학습 데이터의 평균/표준편차가 저장됨 → 나중에 새 분자에 똑같이 적용해야 함
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

# ⭐ 학습은 normalization 된 X_scaled로! (X_sel 원본 ❌)
clf = LogisticRegression(max_iter=5000)
clf.fit(X_scaled, y)
print('학습 정확도:', round(clf.score(X_scaled, y), 3))

선택된 descriptor: ['MinAbsEStateIndex', 'BertzCT', 'Chi0', 'Chi1', 'PEOE_VSA7', 'SlogP_VSA6', 'HeavyAtomCount', 'MolMR', 'fr_C_O_noCOO', 'fr_ester']
학습 정확도: 0.872


## B-2. 모델 저장하기

Logistic Regression은 normalization이 필요한 모델이라 **세 가지**를 같이 저장해야 해.

1. **`cols`** — SelectKBest가 고른 feature 이름. 새 분자에서 **같은 컬럼**을 같은 순서로 뽑아야 함.
2. **`scaler`** — 학습 때 fit된 StandardScaler. 학습 데이터의 평균/표준편차 값이 들어 있음.
3. **`clf`** — 학습된 LogisticRegression.

세 개를 **딕셔너리**로 묶으면 파일 하나로 깔끔하게 저장 가능.

In [2]:
import joblib

# 세 가지를 딕셔너리로 묶기
saved = {
    'features': list(cols),   # feature 이름 (리스트로 변환)
    'scaler': scaler,         # 학습 때 fit한 StandardScaler
    'model': clf              # 학습된 LogisticRegression
}

# 한 파일로 저장
joblib.dump(saved, 'model_logreg.joblib')
print('저장 완료: model_logreg.joblib')

저장 완료: model_logreg.joblib


## B-3. 모델 불러와서 잘 저장됐는지 확인

저장이 잘 됐는지 확인하는 가장 쉬운 방법:
**불러와서 같은 X에 다시 예측 → 학습 때 확인한 성능과 동일하면 성공.**

> ⚠️ 불러온 scaler는 **`transform`** 만 호출! `fit_transform`을 다시 부르면 평균/표준편차가 새로 계산되어 학습 때와 달라짐.

In [3]:
import joblib

# 파일 불러오기
loaded = joblib.load('model_logreg.joblib')
features = loaded['features']
scaler   = loaded['scaler']
model    = loaded['model']

print('저장된 feature 개수:', len(features))
print('저장된 모델 종류  :', type(model).__name__)

# 같은 X에 다시 예측 → 학습 때와 같은 정확도가 나와야 함
X_check = X[features]                          # 저장된 feature 이름으로 다시 추출
X_check_scaled = scaler.transform(X_check)     # ⚠️ transform만! fit_transform 아님
y_pred = model.predict(X_check_scaled)

print('불러온 모델 정확도:', round((y_pred == y).mean(), 3))
print('학습 때 정확도와 같으면 저장 성공 ✅')

저장된 feature 개수: 10
저장된 모델 종류  : LogisticRegression
불러온 모델 정확도: 0.872
학습 때 정확도와 같으면 저장 성공 ✅


## B-4. 모델을 이용해서 새로운 분자구조 예측

학습된 모델로 **새로운 분자의 피부 자극 여부를 예측**해 보자.

## 🚨 핵심 — 학습이 normalization 된 데이터로 됐으니, 예측 입력도 똑같이 normalization

**모델은 학습할 때 사용된 데이터 분포를 기억**해. Logistic Regression은 평균 0, 표준편차 1인 값을 보고 가중치를 정했어. 그런데 새 분자의 원본 값(MolWt=200, logP=3.5 …)을 그대로 넣으면 모델 입장에서는 완전히 숫자라 예측 결과가 제대로 나오지 않음.

**예측 순서 (3단계, 순서 중요!)**
1. 학습 때 골랐던 **같은 feature**만 추출 (`features`)
2. 학습 때 fit한 **같은 scaler**로 `transform` ← ⚠️ Logistic Regression은 학습이 normalized로 됐으니 예측도 반드시 normalized
3. 모델로 `predict` / `predict_proba`

> ❌ **하지 말 것**: `StandardScaler().fit_transform(new_X)` — 새 평균/표준편차가 계산되어 학습 때와 다른 스케일이 되어 버림.
> ✅ **해야 할 것**: 저장된 `scaler.transform(new_X)` — 학습 때의 평균/표준편차를 그대로 적용.

In [5]:
from rdkit import Chem
from rdkit.Chem import Descriptors

# 새로 예측할 분자 3개 (이름 + SMILES)
new_mols = {
    'Acetaminophen': 'CC(=O)Nc1ccc(O)cc1',
    'Aspirin':       'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine':      'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
}

# 분자별 descriptor 계산 → DataFrame으로 정리
rows = []
for name, smi in new_mols.items():
    mol = Chem.MolFromSmiles(smi)        # SMILES → Mol 객체
    desc = Descriptors.CalcMolDescriptors(mol)
    rows.append(desc)

new_X = pd.DataFrame(rows)

# 1) 학습 때 골랐던 같은 feature만 같은 순서로 추출
new_X_sel = new_X[features]

# 2) 학습 때 fit한 같은 scaler로 변환 (transform만! fit_transform 아님)
new_X_scaled = scaler.transform(new_X_sel)

# 3) 예측
new_pred = model.predict(new_X_scaled)
new_prob = model.predict_proba(new_X_scaled)

print('분자          :', list(new_mols.keys()))
print('예측 label    :', list(new_pred))
print('예측 확률(0,1):')
print(new_prob.round(3))

분자          : ['Acetaminophen', 'Aspirin', 'Caffeine']
예측 label    : [np.int64(0), np.int64(0), np.int64(1)]
예측 확률(0,1):
[[0.954 0.046]
 [0.977 0.023]
 [0.301 0.699]]


---
# Part C. ✏️ 직접 해보기 — ANN, SVM, Tree, RF

Logistic Regression에서 했던 패턴 그대로 다른 모델도 저장해 봐.

**저장 패턴 요약**

```python
# normalization 필요한 모델 (LogReg, ANN, SVM)
saved = {'features': ..., 'scaler': ..., 'model': ...}

# normalization 불필요한 모델 (Tree, RF)
saved = {'features': ..., 'model': ...}    # scaler 없음
```

## ✏️ Quiz 1 — ANN (MLPClassifier) 저장

ANN은 normalization 필요 → **scaler도 같이 저장**.

**힌트**
- `from sklearn.neural_network import MLPClassifier`
- `MLPClassifier(hidden_layer_sizes=(??,), ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장 파일명: `model_ann.joblib`
- B-2 ~ B-4 패턴 그대로

In [18]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import joblib

selector = SelectKBest(f_classif, k=50)

# (1) 학습 — SelectKBest로 고른 cols, X_sel 그대로 사용
# scaler 새로 만들기 (학습 데이터에 fit)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

# MLPClassifier 학습
ann = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation='logistic',
    max_iter=5000
)

ann.fit(X_scaled, y)

print('ANN accuracy:',
      ann.score(X_scaled, y))


# (2) 저장 — features + scaler + model
# saved = {'features': ..., 'scaler': ..., 'model': ...}
# joblib.dump(saved, 'model_ann.joblib')
saved = {
    'features': list(cols),
    'scaler': scaler,
    'model': ann
}

joblib.dump(saved, 'model_ann.joblib')

print('저장 완료')

# (3) 불러와서 정확도 확인
loaded = joblib.load('model_ann.joblib')

features = loaded['features']
scaler = loaded['scaler']
model = loaded['model']

X_check = X[features]

X_check_scaled = scaler.transform(X_check)

y_pred = model.predict(X_check_scaled)

print('불러온 ANN accuracy:',
      (y_pred == y).mean())

# (4) 마지막 3개 분자에 예측
new_X_sel = new_X[features]

new_X_scaled = scaler.transform(new_X_sel)

pred = model.predict(new_X_scaled)

prob = model.predict_proba(new_X_scaled)

print('예측 결과:')
print(pred)

print()

print('예측 확률:')
print(prob.round(3))

ANN accuracy: 0.9743589743589743
저장 완료
불러온 ANN accuracy: 0.9743589743589743
예측 결과:
[0 0 1]

예측 확률:
[[0.999 0.001]
 [1.    0.   ]
 [0.    1.   ]]


## ✏️ Quiz 2 — SVM 저장

SVM도 normalization 필요 → **scaler도 같이 저장**.

**힌트**
- `from sklearn.svm import SVC`
- `SVC(C=??, ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장 파일명: `model_svm.joblib`

In [17]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import joblib

selector = SelectKBest(f_classif, k=15)

# (1) 학습
# scaler 생성
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_sel)

# SVM 학습
svm = SVC(
    C=0.1,               # ← grid search 최적값
    kernel='linear',     # ← grid search 최적값
    probability=True
)

svm.fit(X_scaled, y)

print('SVM accuracy:',
      svm.score(X_scaled, y))

# (2) 저장 — features + scaler + model → 'model_svm.joblib'
saved = {
    'features': list(cols),
    'scaler': scaler,
    'model': svm
}

joblib.dump(saved, 'model_svm.joblib')

print('저장 완료')

# (3) 불러와서 정확도 확인
loaded = joblib.load('model_svm.joblib')

features = loaded['features']
scaler = loaded['scaler']
model = loaded['model']

X_check = X[features]

X_check_scaled = scaler.transform(X_check)

y_pred = model.predict(X_check_scaled)

print('불러온 SVM accuracy:',
      (y_pred == y).mean())

# (4) 마지막 3개 분자에 예측
new_X_sel = new_X[features]

new_X_scaled = scaler.transform(new_X_sel)

pred = model.predict(new_X_scaled)

prob = model.predict_proba(new_X_scaled)

print('예측 결과:')
print(pred)

print()

print('예측 확률:')
print(prob.round(3))

SVM accuracy: 0.7948717948717948
저장 완료
불러온 SVM accuracy: 0.7948717948717948
예측 결과:
[0 0 0]

예측 확률:
[[0.767 0.233]
 [0.825 0.175]
 [0.598 0.402]]


ANN은 학습 데이터 정확도는 높았으나(train=0.974), 교차검증 성능(CV=0.768)은 SVM(CV=0.800)보다 낮았다. 또한 예측 확률이 극단적으로 나타나 과적합 가능성이 관찰되었다. 반면 SVM은 비교적 안정적인 확률 분포와 더 높은 교차검증 성능을 보여 최종 모델로 선택하였다.

## ✏️ Quiz 3 — Decision Tree 저장

트리는 normalization **불필요** → 저장할 때 **scaler 빼기**.
예측할 때도 `scaler.transform` 호출 ❌. `X_sel` 그대로 모델에 넣음.

**힌트**
- `from sklearn.tree import DecisionTreeClassifier`
- `DecisionTreeClassifier(max_depth=?, ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장 파일명: `model_tree.joblib`
- 저장 딕셔너리: `{'features': ..., 'model': ...}` (scaler 없음)

In [13]:
from sklearn.tree import DecisionTreeClassifier
import joblib

# (1) 학습 — X_sel 그대로 (scaling 없이)
tree = DecisionTreeClassifier(
    max_depth=3,   # ← 네 최적값으로 수정 가능
    random_state=1
)

tree.fit(X_sel, y)

print('Tree accuracy:',
      tree.score(X_sel, y))

# (2) 저장 — features + model만 → 'model_tree.joblib'
# saved = {'features': ..., 'model': ...}
saved = {
    'features': list(cols),
    'model': tree
}

joblib.dump(saved, 'model_tree.joblib')

print('저장 완료')

# (3) 불러와서 정확도 확인
# X_check = X[features]   # scaler.transform 없음!
# y_pred = model.predict(X_check)
loaded = joblib.load('model_tree.joblib')

features = loaded['features']
model = loaded['model']

X_check = X[features]

y_pred = model.predict(X_check)

print('불러온 Tree accuracy:',
      (y_pred == y).mean())

# (4) 마지막 3개 분자에 예측
# new_X[features] 그대로 model.predict — scaler 없음
new_X_sel = new_X[features]

pred = model.predict(new_X_sel)

prob = model.predict_proba(new_X_sel)

print('예측 결과:')
print(pred)

print()

print('예측 확률:')
print(prob.round(3))

Tree accuracy: 0.9487179487179487
저장 완료
불러온 Tree accuracy: 0.9487179487179487
예측 결과:
[0 0 0]

예측 확률:
[[1. 0.]
 [1. 0.]
 [1. 0.]]


## ✏️ Quiz 4 — Random Forest 저장

Decision Tree와 동일하게 normalization **불필요**.

**힌트**
- `from sklearn.ensemble import RandomForestClassifier`
- `RandomForestClassifier(n_estimators=??, max_depth=?, ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장 파일명: `model_rf.joblib`
- Tree와 같은 패턴 (scaler 없음)

In [14]:
from sklearn.ensemble import RandomForestClassifier
import joblib

# (1) 학습
rf = RandomForestClassifier(
    n_estimators=100,   # ← 네 최적값으로 수정 가능
    max_depth=5,        # ← 네 최적값으로 수정 가능
    random_state=1
)

rf.fit(X_sel, y)

print('RF accuracy:',
      rf.score(X_sel, y))

# (2) 저장 — features + model → 'model_rf.joblib'
saved = {
    'features': list(cols),
    'model': rf
}

joblib.dump(saved, 'model_rf.joblib')

print('저장 완료')

# (3) 불러와서 정확도 확인
loaded = joblib.load('model_rf.joblib')

features = loaded['features']
model = loaded['model']

X_check = X[features]

y_pred = model.predict(X_check)

print('불러온 RF accuracy:',
      (y_pred == y).mean())

# (4) 마지막 3개 분자에 예측
new_X_sel = new_X[features]

pred = model.predict(new_X_sel)

prob = model.predict_proba(new_X_sel)

print('예측 결과:')
print(pred)

print()

print('예측 확률:')
print(prob.round(3))

RF accuracy: 1.0
저장 완료
불러온 RF accuracy: 1.0
예측 결과:
[0 0 0]

예측 확률:
[[0.95 0.05]
 [0.97 0.03]
 [0.84 0.16]]


## ✏️ Quiz 5 — XGBoost 저장

XGBoost도 tree 기반 모델 → normalization **불필요**.

⚠️ **XGBoost는 joblib 대신 자체 포맷(`.json`) 사용 권장**
- joblib(pickle)으로 저장하면 XGBoost 버전이 바뀔 때 호환성 문제가 생길 수 있음.
- XGBoost가 제공하는 `model.save_model('파일.json')` / `model.load_model('파일.json')`이 표준이고 버전 안전.
- 단, `save_model`은 **모델만** 저장 → `features`는 별도 파일로 따로 보관해야 함.

**힌트**
- `from xgboost import XGBClassifier`
- `XGBClassifier(n_estimators=??, max_depth=?, ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장: `model.save_model('model_xgb.json')` + `joblib.dump(list(features), 'features_xgb.joblib')`
- 불러오기: `model = XGBClassifier(); model.load_model('model_xgb.json')`

In [15]:
from xgboost import XGBClassifier
import joblib

# (1) 학습 — X_sel 그대로 (scaling 없이)
xgb = XGBClassifier(
    n_estimators=100,   # ← 네 최적값으로 수정 가능
    max_depth=3,        # ← 네 최적값으로 수정 가능
    random_state=1
)

xgb.fit(X_sel, y)

print('XGB accuracy:',
      xgb.score(X_sel, y))

# (2) 저장 — XGBoost 자체 포맷 (.json)
# model.save_model('model_xgb.json')          # 모델은 XGBoost 전용 포맷
# joblib.dump(list(features), 'features_xgb.joblib')   # features는 따로
xgb.save_model('model_xgb.json')

joblib.dump(list(cols),
            'features_xgb.joblib')

print('저장 완료')

# (3) 불러와서 정확도 확인
# model = XGBClassifier()
# model.load_model('model_xgb.json')
# features = joblib.load('features_xgb.joblib')
# X_check = X[features]   # scaler.transform 없음!
# y_pred = model.predict(X_check)
model = XGBClassifier()

model.load_model('model_xgb.json')

features = joblib.load('features_xgb.joblib')

X_check = X[features]

y_pred = model.predict(X_check)

print('불러온 XGB accuracy:',
      (y_pred == y).mean())

# (4) 새 분자 3개에 예측 (Acetaminophen, Aspirin, Caffeine)
# B-4와 동일하게 RDKit으로 descriptor 계산 → features 추출 → model.predict
# scaler.transform은 호출하지 않음

new_X_sel = new_X[features]

pred = model.predict(new_X_sel)

prob = model.predict_proba(new_X_sel)

print('예측 결과:')
print(pred)

print()

print('예측 확률:')
print(prob.round(3))

XGB accuracy: 0.9743589743589743
저장 완료
불러온 XGB accuracy: 0.9743589743589743
예측 결과:
[0 0 0]

예측 확률:
[[0.981 0.019]
 [0.874 0.126]
 [0.549 0.451]]


---
## 정리 — 기계학습 모델 개발의 최종 결과물

**저장 / 불러오기 핵심 명령**
```python
joblib.dump(객체, '파일.joblib')   # 저장 (대부분 모델)
객체 = joblib.load('파일.joblib')   # 불러오기

# XGBoost는 자체 포맷 (.json) 사용
model.save_model('파일.json')
model.load_model('파일.json')
```

**같이 저장해야 할 것**
1. `features` — SelectKBest로 고른 컬럼 이름 (새 분자에서 같은 순서로 추출)
2. `scaler` — normalization 쓰는 모델만 (LogReg / ANN / SVM)
3. `model` — 학습된 모델 본체

**새 분자 예측 3단계**
1. `new_X[features]` — 같은 feature만 같은 순서로
2. `scaler.transform(...)` — ⚠️ `fit_transform` 아니고 **`transform`**
3. `model.predict(...)`

**모델별 정리**

| 모델 | 저장 포맷 | scaler 저장? | 예측 시 transform 호출? |
|---|---|---|---|
| Logistic Regression | `.joblib` | ✅ | ✅ |
| ANN (MLP) | `.joblib` | ✅ | ✅ |
| SVM | `.joblib` | ✅ | ✅ |
| Decision Tree | `.joblib` | ❌ | ❌ |
| Random Forest | `.joblib` | ❌ | ❌ |
| XGBoost | `.json` (+ features는 별도 `.joblib`) | ❌ | ❌ |

> ✅ 5개 quiz를 다 끝냈다면, 디렉터리에 `model_logreg.joblib`, `model_ann.joblib`, `model_svm.joblib`, `model_tree.joblib`, `model_rf.joblib`, `model_xgb.json`, `features_xgb.joblib` 일곱 개 파일이 생겨 있어야 함.